In [ ]:
from pathlib import Path
from typing import Union
import logging

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

logger = logging.getLogger()

In [ ]:
%%time
spark = create_spark_session(
    aws_profile="admin-user",
    start_spark_cluster=True,
    executor_instances=64,
    executor_memory="32g",
    executor_cores=4
)
ev = teehr.RemoteReadWriteEvaluation(
    spark=spark,
    enable_spark_proxy=True
)

In [ ]:
GROUPBY = ["configuration_name", "variable_name"]
OUTPUT_TABLE_NAME = "configurations_summary"

In [ ]:
def summarize_configurations_by_timeseries_type(
    ev: teehr.Evaluation,
    timeseries_type: str
) -> None:
    """Summarize configurations by timeseries type (primary vs secondary)."""
    group_by_clause = ", ".join([f"{c}" for c in GROUPBY])

    member_cols = ""
    if timeseries_type == "secondary":
        member_cols = """
            ,
            COUNT(DISTINCT member)  AS n_members,
            COLLECT_SET(member)     AS members
        """
    else:
        member_cols = """
            ,
            CAST(NULL AS BIGINT)    AS n_members,
            CAST(NULL AS ARRAY<STRING>) AS members
        """
    logger.info(
        f"Summarizing configurations by {timeseries_type} timeseries into a spark dataframe..."
    )
    return ev.spark.sql(f"""
        WITH agg AS (
            SELECT
                configuration_name,
                variable_name,
                unit_name,
                MIN(reference_time)             AS min_reference_time,
                MAX(reference_time)             AS max_reference_time,
                MIN(value_time)                 AS min_value_time,
                MAX(value_time)                 AS max_value_time,
                approx_count_distinct(location_id)    AS n_locations
                {member_cols}
            FROM iceberg.teehr.{timeseries_type}_timeseries
            GROUP BY unit_name, {group_by_clause}
        )
        SELECT
            agg.*,
            c.description,
            c.timeseries_type,
            SPLIT(agg.configuration_name, '_')[0] AS location_id_prefix
        FROM agg
        JOIN iceberg.teehr.configurations c
            ON c.name = agg.configuration_name
    """)

In [ ]:
primary_sdf = summarize_configurations_by_timeseries_type(
    ev=ev,
    timeseries_type="primary"
)
secondary_sdf = summarize_configurations_by_timeseries_type(
    ev=ev,
    timeseries_type="secondary"
)
result_sdf = primary_sdf.unionByName(secondary_sdf)

In [ ]:
%%time
ev._write.to_warehouse(
    source_data=result_sdf,
    table_name=OUTPUT_TABLE_NAME,
    write_mode="create_or_replace"
)